# cvprofiles — construct-validity profiles

An open methods package for **construct-validity profiles**: partial identification over a finite menu of measurement functions, disciplined by a researcher-authored nomological network. The engine returns an admissible measurement set `M*` and a construct-identified range `[L,U]` for a target functional `beta`.

Everything in this notebook is generated inline — no repository files needed, only the installed package. It shows SCORE → RESTRICT → IDENTIFY → REPORT, plus the empty-set contrast.


## Part 1 — synthetic walk-through

In [ ]:
from __future__ import annotations
import json
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

import cvprofiles
from cvprofiles.pipeline import run_profile

print('cvprofiles', cvprofiles.__version__)

### Build a synthetic scores matrix

One row per unit; one column per measure (`m_*`), plus an auxiliary (`v_aux`) and an outcome (`y`). The researcher supplies the menu — the engine never invents measures.

In [ ]:
rng = np.random.default_rng(42)
n = 200
v_aux = rng.normal(size=n)
m_good = 0.8 * v_aux + 0.6 * rng.normal(size=n)   # strongly aligned with the aux
m_weak = 0.45 * v_aux + 0.9 * rng.normal(size=n)  # weakly aligned
m_slop = -0.5 * v_aux + 1.0 * rng.normal(size=n)  # wrong sign vs the aux
y = 0.5 * m_good + rng.normal(size=n)

scores = pd.DataFrame({
    'unit_id': [f'u{i:03d}' for i in range(n)],
    'm_good': m_good,
    'm_weak': m_weak,
    'm_slop': m_slop,
    'v_aux': v_aux,
    'y': y,
})

roles = {
    'unit_id': 'unit_id',
    'measures': ['m_good', 'm_weak', 'm_slop'],
    'aux': ['v_aux'],
    'outcome': 'y',
    'diagnostic': [],
}

# Nomological network R: admissible measures must correlate with v_aux >= 0.35
# and correlate positively with magnitude >= 0.10.
network = {
    'schema_version': '1',
    'name': 'tutorial_synthetic',
    'delta': 0.0,
    'restrictions': [
        {'id': 'r_corr_min_aux', 'type': 'corr_min', 'theta': 0.35, 'params': {'variable': 'v_aux'}},
        {'id': 'r_corr_sign_aux', 'type': 'corr_sign', 'theta': 0.10, 'params': {'variable': 'v_aux', 'sign': 1}},
    ],
}

beta = {'schema_version': '1', 'type': 'corr_y', 'outcome': 'y', 'params': {}}

### Run a profile

Write the inputs to disk (the engine reads files), then run the full SCORE → RESTRICT → IDENTIFY → REPORT composition. Note the division of labor: the scores, the roles, the network, and the target functional are all *researcher-owned inputs* — the engine never invents a measure and never tunes a threshold.

In [ ]:
work = Path(tempfile.mkdtemp(prefix='cvp_tutorial_'))
scores.to_csv(work / 'scores.csv', index=False)
(work / 'roles.json').write_text(json.dumps(roles))
(work / 'network.yaml').write_text(yaml.safe_dump(network))
(work / 'beta.yaml').write_text(yaml.safe_dump(beta))

result = run_profile(
    scores=work / 'scores.csv',
    roles=work / 'roles.json',
    network=work / 'network.yaml',
    beta=work / 'beta.yaml',
    out_dir=work / 'run',
    seed=0,
    title='Synthetic walk-through',
)

print('run_id :', result.run_id)
print('M*     :', result.identify.admissible)
print('[L,U]  :', result.identify.range_L, result.identify.range_U)
print('rejected:', result.identify.rejected)
print('report :', result.report.html_path)

**Reading the output.**

- `run_id` is a deterministic fingerprint of the frozen inputs (scores, roles, network, β, seed, package version). Same inputs → same id, so any reported number can be audited by re-running.
- `M*` is the **admissible menu**: the measures whose sample slacks clear every declared bar. Here the designed-valid pair survives.
- `[L,U]` is the **construct-identified range**: the min and max of β across survivors *only*. Rejected measures never enter it — that is the whole discipline.
- `rejected` maps each failed measure to the restrictions that bound it. This is not an error log; it is the audit trail saying *which theoretical claim each candidate contradicted*.
- Every run also writes `report.html` (human-readable) and `report.json` (machine-complete) under `out_dir` — a non-coder can steer from the HTML alone.

### Read the rejection: the slack table

Filtering is only half the story — the other half is *accounting*. For every measure and every restriction, the engine records a **slack**: a signed distance to the declared bar. Slack ≥ 0 clears the restriction; the magnitude says by how much. A slack near zero flags a marginal survivor worth worrying about before, not after, publication.

In [ ]:
slacks = result.identify.slacks.round(3)

print('Sample slacks s_r(m_j) — rows: measures, columns: restrictions')
display(slacks)

print()
print('Per-measure beta values:')
for m, b in result.identify.beta_values.items():
    star = ' (admissible)' if m in result.identify.admissible else ''
    print(f'  {m}: {b:+.3f}{star}')

`m_slop` fails *both* restrictions, and deeply — its correlation with the auxiliary runs the wrong way, violating the sign restriction by ~0.52 and the level bar by ~0.77. Its β never touches `[L,U]`. Meanwhile `m_weak` clears both bars but with a thin margin on the level restriction (`r_corr_min_aux`): a different sample could plausibly flip it. The protocol makes that fragility visible *before* anyone builds conclusions on the measure.

### Empty admissible set is a clean result

If the theory + data reject the whole menu, that is a finding, not a crash — the run exits 0 and the report explains the binding bars. An empty `M*` says *"no measurement in this menu is entitled to the interpretation under this theory"* — exactly what you want to learn before, not after, the paper is written. The engine will never quietly loosen thresholds to avoid this outcome.

In [ ]:
network_harsh = {
    **network,
    'name': 'tutorial_harsh',
    'restrictions': [
        {'id': 'r_corr_min_aux', 'type': 'corr_min', 'theta': 0.99, 'params': {'variable': 'v_aux'}},
    ],
}
(work / 'network_harsh.yaml').write_text(yaml.safe_dump(network_harsh))

harsh = run_profile(
    scores=work / 'scores.csv',
    roles=work / 'roles.json',
    network=work / 'network_harsh.yaml',
    beta=work / 'beta.yaml',
    out_dir=work / 'harsh',
    seed=0,
    title='Empty-set contrast',
)

print('empty:', harsh.identify.empty)
print('M*   :', harsh.identify.admissible)
print('[L,U]:', harsh.identify.range_L, harsh.identify.range_U)

### The width of [L,U] is information, not embarrassment

Finally, watch what happens when the theory is tightened instead of loosened. Raise the level bar so only the strongly aligned measure survives:

In [ ]:
network_tight = {
    **network,
    'name': 'tutorial_tight',
    'restrictions': [
        {'id': 'r_corr_min_aux', 'type': 'corr_min', 'theta': 0.50, 'params': {'variable': 'v_aux'}},
        {'id': 'r_corr_sign_aux', 'type': 'corr_sign', 'theta': 0.10, 'params': {'variable': 'v_aux', 'sign': 1}},
    ],
}
(work / 'network_tight.yaml').write_text(yaml.safe_dump(network_tight))

tight = run_profile(
    scores=work / 'scores.csv',
    roles=work / 'roles.json',
    network=work / 'network_tight.yaml',
    beta=work / 'beta.yaml',
    out_dir=work / 'tight',
    seed=0,
    title='Tightened network',
)

print('M*     :', tight.identify.admissible)
print('[L,U]  :', tight.identify.range_L, tight.identify.range_U)

With the stricter bar, `m_weak` drops out and the range collapses to a single value: sharp, but resting entirely on one measure. That is the trade the protocol makes explicit:

| Network | Admissible menu | Range | Reading |
|---|---|---|---|
| baseline (θ = 0.35) | `m_good`, `m_weak` | `[0.138, 0.346]` | robust across two measures, honest spread |
| tightened (θ = 0.50) | `m_good` | point range | sharp, but single-measure-dependent |
| harsh (θ = 0.99) | ∅ | — | theory rejects the whole menu |

A narrow range built on one fragile survivor and a wide range built on many defensible ones are different scientific objects — the range plus the slack table tell you which one you have. Neither bootstrap standard errors nor specification curves surface this distinction, because neither asks whether the *measure* was valid in the first place.

## What to look at next

- Each run writes an audit trail: `report.html` (human-readable), `report.json` (machine-complete), slacks, the admissible set, the range, and — when requested — bootstrap / θ-grid / δ-grid diagnostics and the anchors audit.
- Empty `M*` and wide `[L,U]` are scientific features: they quantify measurement fragility under the stated theory.
- The engine is score-agnostic and model-free. Scoring, the menu, and the nomological network are researcher-owned — the columns here could have been WVS items, LLM log-probs, or IRT trait scores without changing a line of engine code.
- Thresholds come from the researcher's theory and literature, never tuned to the data.
- Deeper tours: full evaluator + diagnostic stack (`cvprofiles_diagnostics_tour.ipynb`), IRT as an upstream scorer (`cvprofiles_irt_scoring_tutorial.ipynb`), OVB sensitivity on a survivor (`cvprofiles_sensemakr_tutorial.ipynb`), and the flagship WVS/GPS patience application (`cvprofiles_wvs_gps_inputs.ipynb`, `evals/wvs_gps_preferences/`).